# Fit MIMIC image embeddings

Load a serialized vision dataset, fit MIMIC on the flattened image rows, compute embeddings with `transform`, and save those embeddings for later visualization.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "vision" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ROOT_SRC = PROJECT_ROOT / "src"
VISION_SRC = PROJECT_ROOT / "vision" / "src"
for src_dir in [str(ROOT_SRC), str(VISION_SRC)]:
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

from mimic import MIMIC
from mimic_vision import (
    VisionEmbedding,
    load_serialized_vision_dataset,
    save_vision_embedding,
)

In [ ]:
DATASET_FILE = "mnist_train_n400_classes-3-8_28x28.pkl"
DATASET_DIR = PROJECT_ROOT / "vision" / "data" / "serialized"
EMBEDDING_DIR = PROJECT_ROOT / "vision" / "data" / "embeddings"

MODE = "direct"
CAPACITY = 0.0
RANDOM_STATE = 0
BOOTSTRAP = False
FEATURE_N_JOBS = 1

In [ ]:
dataset = load_serialized_vision_dataset(DATASET_FILE, input_dir=DATASET_DIR)
dataset.X.shape, dataset.images.shape, dataset.y.value_counts().sort_index()

In [ ]:
columns = {
    "regression": list(dataset.X.columns),
    "classification": [],
    "ignore": [],
}

model = MIMIC(
    columns=columns,
    mode=MODE,
    capacity=CAPACITY,
    bootstrap=BOOTSTRAP,
    feature_n_jobs=FEATURE_N_JOBS,
    random_state=RANDOM_STATE,
)
model.fit(dataset.X)

In [ ]:
embeddings = model.transform(dataset.X)
embeddings.shape

In [ ]:
embedding_artifact = VisionEmbedding(
    dataset_file=DATASET_FILE,
    embeddings=embeddings,
    mode=MODE,
    capacity=CAPACITY,
    random_state=RANDOM_STATE,
)
saved_embedding_path = save_vision_embedding(embedding_artifact, output_dir=EMBEDDING_DIR)
saved_embedding_path.name